In [14]:
import os
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain.tools import tool
from deepagents import create_deep_agent, FilesystemPermission, CompiledSubAgent
from langchain.agents import  create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from deepagents.backends import FilesystemBackend,StateBackend,StoreBackend,CompositeBackend
from langchain.chat_models import init_chat_model
from tools.get_weather import  get_weather
from tools.search_document import search_document
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
memory = InMemorySaver()
store = InMemoryStore()

In [16]:
# model = init_chat_model('openai:gpt-4')
ACCOUNT_ID = os.environ["CLOUDFLARE_ACCOUNT_ID"]
API_TOKEN  = os.environ["CLOUDFLARE_API_TOKEN"]

# --- 1. Model -------------------------------------------------------------
model = ChatOpenAI(
    model="@cf/zai-org/glm-5.2",          # Workers AI model id
    base_url=f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/v1",
    api_key=API_TOKEN,
    temperature=0,
)

In [17]:
# from pathlib import  Path
# from langchain_core.documents import Document

# source_pth = Path("./files/office.txt")

# docs = [Document(page_content=source_pth.read_text(encoding='utf-8'),metadata={'source':str(source_pth)})]

In [18]:
# from langchain_text_splitters import  RecursiveCharacterTextSplitter

# splits = RecursiveCharacterTextSplitter(
#     chunk_size=500, chunk_overlap=100
# )

# chunks = splits.split_documents(docs)

In [19]:
# from langchain_openai import OpenAIEmbeddings

# embedding = OpenAIEmbeddings()

In [20]:
# from langchain_core.vectorstores import  InMemoryVectorStore

# vectorstore = InMemoryVectorStore.from_documents(chunks,embedding)

In [21]:
# retriever = vectorstore.as_retriever()

In [22]:
# @tool
# def search_document(query:str)-> str:
#     """return relavant passage from the loaded document."""
#     found = retriever.invoke(query)
#     return "\n\n".join(doc.page_content for doc in found)

In [23]:
sub_agent_retriever = create_agent(
    model=model,
    tools=[search_document],
    system_prompt=(
        "You answer questions about the loaded document. "
        "Always use search_document to ground your answers."
    ),
)

In [24]:

# result = sub_agent_retriever.invoke({'messages': HumanMessage(content='how many people work in meridian analytics and their names?')})
# print(result['messages'][-1].content)


In [25]:
retriever_agent = CompiledSubAgent(
    name= "retriever agent",
    description = "Specilized agent for retrieving documents",
    runnable = sub_agent_retriever
)

In [26]:
# @tool
# def get_weather(city:str):
#     """"get weather of a city"""
#     return f"The weather of  {city} is 18 degrees celcius and rainy"

In [27]:
backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": StoreBackend(namespace=lambda _rt: ("use_one",)),
        "/files/": FilesystemBackend(root_dir='./files/',virtual_mode=True),
        "/skills/": FilesystemBackend(root_dir='./skills/', virtual_mode=True),
    }
)


In [28]:
agent = create_deep_agent(
    model=model,
    tools=[get_weather],
    system_prompt= "Use retriever_agent subagent to answer questions about the loaded document.",
    checkpointer=memory,
    store=store,
    memory=['/memories/AGENTS.md'],
    backend=backend,
    skills=["/skills/"],
    subagents = [retriever_agent],
    permissions=[
    FilesystemPermission(
        operations=["write"],
        paths=["/skills/**"],
        mode="deny",
        ),
    ],
)

In [29]:
# agent = create_agent(
#     model=model,
#     tools=[get_weather],
#     system_prompt="You are a friendly assistant",
#     checkpointer=memory
# )

In [30]:
config = {'configurable':{'thread_id':'1'}}
config2 = {'configurable':{'thread_id':'2'}}

In [34]:
result = agent.invoke({'messages':HumanMessage(content="what is Meridian Analytics and what does priya do there and name her 2 collegues and their designations in short?")},config=config2)
print(result['messages'][-1].content)

## Meridian Analytics
A mid-sized software company in Portland, Oregon that builds **forecasting software for grocery retailers**. Its flagship product is **PulseGrid**.

## What Priya Does
**Priya Raman** is the **Chief Executive Officer (CEO)** and co-founder of Meridian Analytics. She still writes code on Friday afternoons and is known for ending all-hands meetings with the phrase, *"Ship something you'd be proud to explain."*

## Two Colleagues and Their Designations
1. **Daniel Okoye** — Chief Technology Officer (CTO)
2. **Sarah Whitfield** — Director of Engineering
